In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-mini", temperature=0.2)

### Tools

In [4]:
from langchain.tools import tool
from langchain_experimental.utilities import PythonREPL

/var/folders/9s/5g5zn_g56652vdtfwm8lqjsw0000gn/T/ipykernel_13931/2622403079.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.utilities import PythonREPL


In [7]:
# In Python, REPL stands for Read-Eval-Print Loop. 
# It refers to the interactive Python interpreter (or Python shell) 
# that executes your code line-by-line and provides immediate feedback

python_repl = PythonREPL()

print(python_repl.run("print(1+1)"))
print(python_repl.run("x=5; print(x*2)"))

2

10



In [ ]:
@tool
def python_repl_tool(code: str) -> str:
    """A Python shell.

    Use this to execute python commands.

    Input should be a valid python command, such as "print(1+1)", "x=5; print(x*2)".

    If you want to see the output of a value, you should print it out with `print(...)` function.
    """
    return python_repl.run(code)

### Inbuilt Agent

In [15]:
import pandas as pd
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

# 1. Initialize your OpenAI Chat Model
# Setting temperature to 0 ensures accurate data analysis rather than creative guesses
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

# 2. Load the uploaded CSV file into a pandas DataFrame
# Replace 'your_data.csv' with the path to your actual uploaded file
df = pd.read_csv("/Users/srskshitizgoel/Documents/Learnings/LangChain Course/CH-3/Iris.csv")

# 3. Create the ReAct Pandas Agent
# 'allow_dangerous_code=True' is explicitly required in recent updates 
# to authorize the agent to run local pandas operations.
agent_executor = create_pandas_dataframe_agent(
    llm=llm,
    df=df,
    verbose=True,  # Set to True to see the agent's exact ReAct "Thought -> Action -> Observation" loop
    allow_dangerous_code=True,
    agent_type="tool-calling",
)

# 4. Prompt the agent for schema details and EDA insights
prompt = (
    "Analyze the schema of this dataset. Tell me the column names, data types, "
    "and any missing value counts. Then, provide 3 key EDA insights or trends "
    "you find relevant."
)

print("--- Agent is analyzing the data ---")
response = agent_executor.invoke({"input": prompt})

print("\n--- Final Analysis ---")
print(response["output"])

--- Agent is analyzing the data ---


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "# Show dataframe info, dtypes, missing values, and basic stats\nprint(df.info())\nprint('\\nDtypes and non-null counts:\\n')\nprint(df.dtypes)\nprint('\\nMissing values per column:\\n')\nprint(df.isnull().sum())\nprint('\\nShape:', df.shape)\nprint('\\nUnique value counts (object columns):')\nfor col in df.select_dtypes(include=['object']).columns:\n    print(col, df[col].nunique())\nprint('\\nSummary statistics:')\nprint(df.describe())\n\n# Also give group-wise means for EDA\nprint('\\nGroup means by Species:\\n', df.groupby('Species').mean())\n\n# Correlation matrix\nprint('\\nCorrelation matrix:\\n', df.select_dtypes(include=[float]).corr())\n\n# Top 5 max/min for SepalLengthCm\nprint('\\nTop 3 SepalLengthCm by Species:\\n', df.sort_values('SepalLengthCm', ascending=False)[['Id','SepalLengthCm','Species']].head(3))\nprint('\\nBottom 3 SepalLengthCm by Species:\\n

### Manual Agent 1

In [17]:
import os
import pandas as pd
from typing import Annotated, Sequence
from langchain_core.tools import tool
from langchain.agents import create_agent

# 1. Load your CSV file
# Ensure 'your_data.csv' is in your directory
df = pd.read_csv("/Users/srskshitizgoel/Documents/Learnings/LangChain Course/CH-3/Iris.csv")

# 2. Build a Custom Python Tool that pre-loads the dataframe
# We instantiate a Python REPL environment
repl = PythonREPL()

# Inject the dataframe directly into the REPL's global environment
repl.globals["df"] = df

@tool
def python_agent_tool(code: str) -> str:
    """
    A Python shell to analyze a pandas DataFrame named 'df'. 
    Use this to run commands like df.info(), df.describe(), checking missing values, or executing operations.
    Always use print() to view the results of your code operations.
    """
    try:
        # Run the code string inside the environment containing 'df'
        result = repl.run(code)
        return result if result else "Code executed successfully with no output."
    except Exception as e:
        return f"Error executing code: {str(e)}"

# Pack our custom tool into a list
tools = [python_agent_tool]

# 3. Setup the OpenAI Chat Model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 4. Construct the custom system prompt to guide the agent's behavior
system_prompt = (
    "You are an expert Exploratory Data Analysis (EDA) assistant. "
    "You have access to a pandas DataFrame called 'df'. "
    "Always start your analysis by inspecting the dataframe schema using the tool. "
    "When asked for insights, perform operations, read the observations, and synthesize a well-formatted summary."
)

# 5. Compile our Custom ReAct Agent
# This under the hood builds the graph structure (LLM -> Tool Execution Loop -> Final Response)
app = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

# 6. Invoke the custom agent
query = "Analyze the schema of my dataset and give me 3 interesting insights or findings from the columns."

print("--- Running Custom ReAct Agent Loop ---")
config = {"configurable": {"thread_id": "eda_session_1"}}

# Using stream allows you to see the exact ReAct 'Thought' and 'Tool Calls' in real-time
for chunk in app.stream({"messages": [("user", query)]}, config, stream_mode="values"):
    message = chunk["messages"][-1]
    message.pretty_print()

--- Running Custom ReAct Agent Loop ---
================================ Human Message =================================

Analyze the schema of my dataset and give me 3 interesting insights or findings from the columns.
================================== Ai Message ==================================
Tool Calls:
  python_agent_tool (call_K6HnclUMRjZtFdtwhCThmPNK)
 Call ID: call_K6HnclUMRjZtFdtwhCThmPNK
  Args:
    code: df.info()
================================= Tool Message =================================
Name: python_agent_tool

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             150 non-null    int64  
 1   SepalLengthCm  150 non-null    float64
 2   SepalWidthCm   150 non-null    float64
 3   PetalLengthCm  150 non-null    float64
 4   PetalWidthCm   150 non-null    float64
 5   Species        150 non-null    str    
dtypes: float64(

### Manual Agent 2

In [ ]:
import pandas as pd
from langchain.tools import tool
from langchain.agents import create_agent

# 1. Load multiple dataframes and store them in a dictionary mapping names to data
datasets = {
    "iris": pd.read_csv("/Users/srskshitizgoel/Documents/Learnings/LangChain Course/CH-3/Iris.csv"),
    "titanic": pd.read_csv("/Users/srskshitizgoel/Documents/Learnings/LangChain Course/CH-3/Titanic.csv")
    # We can add more datasets here as needed
}

@tool
def basic_eda(dataset_name: str) -> str:
    """
    Perform basic exploratory data analysis on a specific dataset by its name.
    """

    data_name = dataset_name.lower().strip()
    if data_name not in datasets:
        return f"Dataset '{dataset_name}' not found. Available datasets: {list(datasets.keys())}"
    df = datasets[data_name]

    report = f"""
        Dataset Shape
        -------------
        {df.shape}

        Columns and Data Types
        ----------
        {df.dtypes.to_string()}

        Summary Statistics
        ------------------
        {df.describe(include="all").to_string()}

        Missing Values
        --------------
        {df.isnull().sum().to_string()}

        Duplicate Rows
        --------------
        {df.duplicated().sum()}
        """

    return report

In [31]:
prompt = f"""
You are an expert Exploratory Data Analysis assistant.

You have access to tools that inspect specific datasets.
Currently, you have access to the following datasets in your system:
{", ".join(datasets.keys())}

Whenever the user asks about a dataset, explicitly pass the correct dataset name to the tool.
Explain the results clearly.
"""

tools = [basic_eda]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=prompt
)

inputs = {"messages": [{"role": "user", "content": "Give basic eda for the iris dataset"}]}
for chunk in agent.stream(inputs, stream_mode="values"):
    message = chunk["messages"][-1]
    message.pretty_print()

================================ Human Message =================================

Give basic eda for the iris dataset
================================== Ai Message ==================================
Tool Calls:
  basic_eda (call_XafYKrcBixjntvtfutBMzUy4)
 Call ID: call_XafYKrcBixjntvtfutBMzUy4
  Args:
    dataset_name: iris
================================= Tool Message =================================
Name: basic_eda


        Dataset Shape
        -------------
        (150, 6)

        Data Types
        ----------
        Id                 int64
SepalLengthCm    float64
SepalWidthCm     float64
PetalLengthCm    float64
PetalWidthCm     float64
Species              str

        Summary Statistics
        ------------------
                        Id  SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm      Species
count   150.000000     150.000000    150.000000     150.000000    150.000000          150
unique         NaN            NaN           NaN            NaN           N